# From Pandas to Dask

This notebook introduces Pandas for structured data analysis and then shows how Dask scales the same API to multiple files and larger-than-memory datasets.

In [ ]:
import psutil
import pathlib
import numpy as np
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt
%matplotlib inline

print(f"Available RAM: {psutil.virtual_memory().available / (1024 ** 3):.1f} GB")

## Part 1: Pandas Basics with Simulated Data

We simulate a physics experiment: measuring distance and time with Gaussian uncertainty, then compute derived quantities using vectorized operations.

In [ ]:
n_samples = 1_000
distance_samples_m = np.random.normal(loc=100, scale=10, size=n_samples)
time_samples_s = np.random.normal(loc=10, scale=2, size=n_samples)

df = pd.DataFrame({"distance_m": distance_samples_m, "time_s": time_samples_s})
df.head()

In [ ]:
df.plot.scatter("time_s", "distance_m", xlabel="time [s]", ylabel="distance [m]", grid=True)

### Vectorized Operations

Pandas applies operations to entire columns at C speed — no Python loops needed:

In [ ]:
df["speed_m_per_s"] = df.distance_m / df.time_s
df.speed_m_per_s.hist(bins=50)
plt.xlabel("speed [m/s]")
plt.ylabel("count");

## Part 2: Real-World Data — Weather Station

We download meteorological data from a Polish weather archive and process it first with Pandas, then with Dask.

In [ ]:
# Download weather data from https://meteo.gig.eu (CC BY 4.0)
import requests, re

data_dir = pathlib.Path("meteo")
data_dir.mkdir(exist_ok=True)

base = "https://meteo.gig.eu/archiwum/2025/"
html = requests.get(base).text
txt_files = sorted(re.findall(r'href="(\d{8}\.txt)"', html))
print(f"Found {len(txt_files)} weekly files at {base}")

for fname in txt_files:
    dest = data_dir / fname
    if dest.exists():
        continue
    url = base + fname
    print(f"Downloading {fname}")
    resp = requests.get(url)
    resp.raise_for_status()
    dest.write_bytes(resp.content)

print(f"Data ready: {len(list(data_dir.glob('*.txt')))} files in {data_dir}/")

### Data Schema

The weather station records 36 variables: temperature, humidity, wind, pressure, solar radiation, etc.

In [ ]:
columns = [
    "Date", "Time", "TempOut", "TempHi", "TempLow", "HumOut", "DewPt",
    "WindSpeed", "WindDir", "WindRun", "WindSpeedHi", "WindDirHi", "WindChill",
    "HeatIndex", "THWIndex", "THSWIndex", "Bar", "Rain", "RainRate",
    "SolarRad", "SolarEnergy", "SolarRadHi", "UVIndex", "UVDose", "UVIndexHi",
    "HeatDD", "CoolDD", "TempIn", "HumIn", "DewPtIn", "HeatIn", "ET",
    "WindSamp", "WindTx", "ISSRecept", "ArcInt"
]

### Pandas: Single File

### Why `read_csv` Needs Extra Arguments Here

We intentionally start with the simplest call and then add arguments one by one to see what fails and why.

In [ ]:
# Step 1: naive call (defaults expect comma-separated data)
first_file = next(data_dir.glob("*.txt"))
df_single_naive = pd.read_csv(first_file)
df_single_naive.head()

In [ ]:
# Step 2: show a concrete failure caused by wrong parsing
try:
    df_single_naive["TempOut"].mean()
except Exception as e:
    print(type(e).__name__ + ":", e)

In [ ]:
# Step 3: split on whitespace, but still no column names / metadata handling
try:
    df_sep_only = pd.read_csv(first_file, sep=r"\s+")
    print("Columns inferred by pandas:", list(df_sep_only.columns)[:8])
    print("... total columns:", len(df_sep_only.columns))
    df_sep_only["TempOut"].mean()
except Exception as e:
    print(type(e).__name__ + ":", e)

In [ ]:
# Step 4: add names, but forget to skip metadata rows
try:
    df_with_names = pd.read_csv(
        first_file,
        sep=r"\s+",
        names=columns,
    )
    pd.to_numeric(df_with_names["TempOut"], errors="raise").mean()
except Exception as e:
    print(type(e).__name__ + ":", e)

In [ ]:
# Step 5 (final): full call with all required arguments
df_single = pd.read_csv(
    first_file,
    sep=r'\s+',
    names=columns,
    skiprows=3,
    na_values=["---", "------"],
)
df_single.head()

In [ ]:
df_single.plot.scatter(x="TempOut", y="HumOut", title="Single file")

### Mini Intro: Data Types (dtypes)

Each DataFrame column has a dtype (data type):
- `float64`: real numbers with decimals
- `int64`: integers (no decimals, no NA support in plain NumPy ints)
- `object`/`string`: text-like values

Let us inspect dtypes in `df_single` and group columns by dtype.

In [ ]:
dtype_table = df_single.dtypes.rename("dtype").to_frame()
display(dtype_table)

dtype_groups = {}
for col, dtype in df_single.dtypes.items():
    key = str(dtype)
    dtype_groups.setdefault(key, []).append(col)

for dtype_name in sorted(dtype_groups):
    cols = dtype_groups[dtype_name]
    print(f"\n{dtype_name} ({len(cols)} columns)")
    print(", ".join(cols))

### NA and Special Numeric Values

Floating-point columns can represent special IEEE values:
- `np.nan` (missing / not-a-number)
- `np.inf` and negative infinity (`-np.inf`, historically also called `np.ninf`)
- `+0.0` and `-0.0` (different sign bit, numerically equal)

These do not map directly to regular integer dtype behavior.

In [ ]:
float_examples = pd.Series([1.0, np.nan, np.inf, -np.inf, 0.0, -0.0], dtype="float64")
print(float_examples)

print("\nnp.nan == np.nan ->", np.nan == np.nan)
print("np.isnan(np.nan) ->", np.isnan(np.nan))
print("+0.0 == -0.0 ->", 0.0 == -0.0)
print("np.signbit(+0.0), np.signbit(-0.0) ->", np.signbit(0.0), np.signbit(-0.0))

with np.errstate(divide="ignore", invalid="ignore"):
    print("1 / +0.0 ->", np.divide(1.0, 0.0))
    print("1 / -0.0 ->", np.divide(1.0, -0.0))

### IEEE-754 Bit Representation of Special Values

For `float64`, bits are split into:
- 1 sign bit `s`
- 11 exponent bits `E`
- 52 fraction (mantissa) bits `F`

`2047` is the maximum **encoded** exponent-field value (`E`, all 11 bits set to 1).
- `E = 2047` is reserved for special values (`inf` and `nan`)
- largest exponent for finite **normal** numbers is `E = 2046`

Value construction formulas:

For normal numbers (`1 <= E <= 2046`):

$$
x = (-1)^s \cdot \left(1 + \frac{F}{2^{52}}\right) \cdot 2^{E-1023}
$$

For subnormal numbers (`E = 0`, `F != 0`):

$$
x = (-1)^s \cdot \left(\frac{F}{2^{52}}\right) \cdot 2^{-1022}
$$

Subnormal numbers are very small non-zero values between `0` and the smallest normal number.
- They use `E = 0` and no implicit leading `1` in the significand
- This gives *gradual underflow*: values can get closer to zero smoothly instead of jumping directly from the smallest normal value to `0`

Special cases:
- `E = 0`, `F = 0` -> signed zero (`+0.0` / `-0.0`)
- `E = 2047`, `F = 0` -> infinities (`+inf` / `-inf`)
- `E = 2047`, `F != 0` -> NaN (payload in fraction bits)

In [ ]:
def float64_fields(x):
    u = np.array([x], dtype=np.float64).view(np.uint64)[0]
    sign = int((u >> 63) & 0x1)
    exponent = int((u >> 52) & 0x7FF)
    fraction = int(u & 0x000FFFFFFFFFFFFF)
    bits = f"{u:064b}"
    bits_grouped = f"{bits[0]} | {bits[1:12]} | {bits[12:]}"
    hex64 = f"0x{u:016x}"

    if exponent == 0x7FF and fraction == 0:
        cls = "infinity"
    elif exponent == 0x7FF and fraction != 0:
        cls = "nan"
    elif exponent == 0 and fraction == 0:
        cls = "zero"
    elif exponent == 0 and fraction != 0:
        cls = "subnormal"
    else:
        cls = "normal"

    return {
        "value": x,
        "hex": hex64,
        "bits_grouped": bits_grouped,
        "sign": sign,
        "exp": exponent,
        "frac": fraction,
        "class": cls,
    }

examples = {
    "+0.0": np.float64(0.0),
    "-0.0": np.float64(-0.0),
    "+inf": np.float64(np.inf),
    "-inf": np.float64(-np.inf),
    "nan": np.float64(np.nan),
    "regular (1.5)": np.float64(1.5),
}

rows = []
for label, val in examples.items():
    row = float64_fields(val)
    row["label"] = label
    rows.append(row)

bit_df = pd.DataFrame(rows)[["label", "value", "class", "sign", "exp", "frac", "hex", "bits_grouped"]]
display(bit_df)

print("\nFull bit patterns (sign | exponent | fraction):")
for row in rows:
    print(f"{row['label']:>14}: {row['bits_grouped']}")

In [ ]:
# NumPy int arrays cannot represent NaN/Inf.
arr_int = np.array([10, 20, 30], dtype=np.int64)
print("NumPy int dtype:", arr_int.dtype)

for bad_value in [np.nan, np.inf, -np.inf]:
    try:
        arr_copy = arr_int.copy()
        arr_copy[1] = bad_value
    except Exception as e:
        print(f"Assign {bad_value!r} -> {type(e).__name__}: {e}")

# Pandas plain int64 also cannot be constructed with NaN directly.
try:
    pd.Series([10, np.nan, 30], dtype="int64")
except Exception as e:
    print(f"\nPandas int64 with NaN -> {type(e).__name__}: {e}")

# If you assign NaN later, pandas upcasts to float64.
int_plain = pd.Series([10, 20, 30], dtype="int64")
int_plain.loc[1] = np.nan
print("\nAfter assigning NaN to int64 series, dtype becomes:", int_plain.dtype)
print(int_plain)

# Strategy 1: pandas nullable integer dtype.
int_nullable = pd.Series([10, 20, pd.NA, 40], dtype="Int64")
print("\nNullable Int64 series:")
print(int_nullable)
print("dtype:", int_nullable.dtype)

# Strategy 2: sentinel code in plain int columns (domain-specific convention).
int_sentinel = pd.Series([10, 20, -1, 40], dtype="int64")
missing_mask = int_sentinel.eq(-1)
print("\nSentinel-based series (-1 means missing):")
print(int_sentinel)
print("missing mask:")
print(missing_mask)

### `Int64` (pandas nullable) vs plain `int64`

Both represent integer-like data, but they have different trade-offs.

Capabilities:
- `int64` (NumPy dtype): fastest and compact, but cannot store missing values directly
- `Int64` (pandas extension dtype): supports missing values via `pd.NA`

Memory footprint:
- `int64` usually uses about 8 bytes per value
- `Int64` needs extra storage for the missing-value mask, so it usually takes more memory

Operation speed:
- arithmetic on plain `int64` is usually faster
- arithmetic on `Int64` is often a bit slower because missing-value handling is built in

Let us measure both on the same synthetic data.

In [ ]:
import time

n = 20_000_000
base = pd.Series(np.arange(n, dtype=np.int64), dtype="int64")
nullable_no_na = base.astype("Int64")
nullable_with_na = base.astype("Int64")

# Introduce a small fraction of missing values only in one nullable series.
missing_idx = np.random.choice(n, size=n // 100, replace=False)
nullable_with_na.iloc[missing_idx] = pd.NA

print("dtypes:", base.dtype, nullable_no_na.dtype, nullable_with_na.dtype)

mem_base = base.memory_usage(deep=True)
mem_nullable_no_na = nullable_no_na.memory_usage(deep=True)
mem_nullable_with_na = nullable_with_na.memory_usage(deep=True)
print(f"int64 memory:            {mem_base / (1024**2):.2f} MB")
print(f"Int64 memory (no NA):    {mem_nullable_no_na / (1024**2):.2f} MB")
print(f"Int64 memory (with NA):  {mem_nullable_with_na / (1024**2):.2f} MB")
print(f"memory ratio Int64/int64: {mem_nullable_with_na / mem_base:.2f}x")

def benchmark(op, s, trials=7, repeats=3):
    times = []
    last = None
    for _ in range(trials):
        t0 = time.perf_counter()
        for _ in range(repeats):
            last = op(s)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    times = np.array(times, dtype=float)
    return float(np.median(times)), float(np.min(times)), last

ops = {
    "add_1": lambda s: s + 1,
    "sum": lambda s: s.sum(skipna=True),
}

series_map = {
    "int64": base,
    "Int64 (no NA)": nullable_no_na,
    "Int64 (with NA)": nullable_with_na,
}

rows = []
for op_name, op in ops.items():
    for series_name, s in series_map.items():
        median_t, min_t, _ = benchmark(op, s)
        rows.append({
            "operation": op_name,
            "series": series_name,
            "median_s": median_t,
            "min_s": min_t,
        })

bench_df = pd.DataFrame(rows)
display(bench_df)

print("\nNotes:")
print("- Timings depend on machine and current load.")
print("- Plain int64 is usually fastest for arithmetic when no missing values are needed.")
print("- Int64 adds missing-value capability at some memory and speed overhead.")

### Pandas: All Files (traditional loop + concat)

The sequential approach: load each file, collect DataFrames, concatenate. This requires all data in memory at once.

In [ ]:
%%time
df_list = []
for file_path in data_dir.glob("*.txt"):
    df_tmp = pd.read_csv(file_path, sep=r'\s+', names=columns, skiprows=3, na_values=["---", "------"])
    df_list.append(df_tmp)
df_all = pd.concat(df_list)
print(f"Total rows: {len(df_all):,}")

### Dask: All Files in One Line

Dask reads all files with a glob pattern and creates a lazy DataFrame. No data is loaded until  is called.

In [ ]:
df_dask = dd.read_csv(
    data_dir / "*.txt", sep=r'\s+', names=columns, skiprows=3,
    na_values=["---", "------"], assume_missing=True
)

# Fix integer columns that may have NaN
for col in ["HumIn", "HumOut", "SolarRad", "SolarRadHi"]:
    df_dask[col] = df_dask[col].fillna(-1).astype("int64")

In [ ]:
# This is lazy — just metadata, no data loaded yet
df_dask

### Triggering Computation

 executes the entire task graph and returns a Pandas DataFrame. Use it only when you need the final result.

In [ ]:
# Compute a single statistic (minimal data transfer)
df_dask.TempOut.mean().compute()

In [ ]:
# Materialize full DataFrame for plotting (all data in memory)
df_dask.compute().plot.scatter(x="TempOut", y="HumOut", s=1, title="All files via Dask")

### Saving to HDF5

Binary formats like HDF5 are much faster to read/write than CSV and support compression.

In [ ]:
h5_path = data_dir / "meteo_dask.h5"
df_dask.to_hdf(h5_path, key="df", mode="w")
print(f"Uncompressed: {h5_path.stat().st_size / (1024**2):.1f} MB")

df_dask.to_hdf(h5_path, key="df", mode="w", complevel=5)
print(f"Compressed:   {h5_path.stat().st_size / (1024**2):.1f} MB")

## Exercise 1

**Compute the mean wind speed** using both Pandas () and Dask (). Verify the results match.

In [ ]:
# Exercise 1: compute mean WindSpeed with both Pandas and Dask

pandas_mean = ___  # YOUR CODE: df_all.WindSpeed...
dask_mean = ___    # YOUR CODE: df_dask.WindSpeed...

print(f"Pandas mean wind speed: {pandas_mean:.4f}")
print(f"Dask mean wind speed:   {dask_mean:.4f}")

<details>
<summary><strong>Hint solution (optional)</strong></summary>

Use the same mean operation in both libraries, and remember that Dask is lazy:
- Pandas: `df_all["WindSpeed"].mean()`
- Dask: `df_dask["WindSpeed"].mean().compute()`

You can check agreement with `np.isclose(...)`.

</details>

## Exercise 2

**Scale the physics simulation to 100 million samples.** Regenerate the distance/time DataFrame. What happens to the speed histogram shape? Does it still look Gaussian?

In [ ]:
# Exercise 2: scale to 100M samples

n_samples_large = 100_000_000

# YOUR CODE HERE: create df_large with distance_m and time_s columns
# Then compute speed and plot the histogram with bins=200

<details>
<summary><strong>Hint solution (optional)</strong></summary>

Create two large normal samples (`distance_m`, `time_s`), build a DataFrame, compute speed, and plot:
- `df_large = pd.DataFrame({...})`
- `df_large["speed_m_per_s"] = df_large["distance_m"] / df_large["time_s"]`
- `df_large["speed_m_per_s"].hist(bins=200)`

If 100M does not fit RAM, test first with 10M and then scale up.

</details>